In [49]:
import pandas as pd
from ortools.sat.python import cp_model
import re

In [50]:
# Lê o arquivo Excel sem carregar os dados ainda
arquivo_excel = pd.ExcelFile('semestre.xlsx')
# Imprime os nomes de todas as páginas (abas)
print("Páginas disponíveis:", arquivo_excel.sheet_names)

Páginas disponíveis: ['Página1', 'disciplinas', 'restricoes', 'formato', 'semestre impar', 'semestre par']


In [51]:
disciplinas = pd.read_excel("semestre.xlsx", sheet_name="disciplinas")
disciplinas

,professor,periodo,disciplina,carga horária,blocado
0,Segundo,9,instrumentação,60,não
1,nda,9,segurança do trabalho,60,3t45;5t45
2,Cecilio,9,automação industrial,60,não
3,Sâmara,9,processamento digital de sinais,60,não
4,Sâmara,8,eletrônica analógica,60,não
5,Sâmara,8,lab de eletrônica analógica,30,não
6,Cecilio,8,controle 2,60,não
7,Mabel,8,banco de dados,60,2t45;4t45
8,Huliane,8,engenharia de software,60,2t23;4t23
9,Walber,8,dependabilidade e segurança,60,6t2345


In [52]:
restricoes = pd.read_excel("semestre.xlsx", sheet_name="restricoes")
restricoes

,peso,restrições
0,6,Obrigatório ofertar todas as cadeiras do semes...
1,5,Obrigatório carga horária de 30 = uma aula na ...
2,5,Obrigatório carga horária de 60 = quatro horár...
3,5,Obrigatório carga horária de 90 = seis horári...
4,5,"Turno manhã: M2, M3, M4, M5"
5,5,"Turno tarde: T1, T2, T3, T4, T5, T6"
6,4,cursar todas as disciplinas do 9 em um turno e...
7,4,"agrupar o maximo de cadeiras por turno (manhã,..."
8,4,professores ofertarem as disciplinas associada...
9,4,ser possivel cursar todas as disciplinas do 8 ...


In [53]:
formato = pd.read_excel("semestre.xlsx", sheet_name="formato")
formato

,Coluna 1,SEGUNDA,TERÇA,QUARTA,QUINTA,SEXTA
0,M2,disponivel,disponivel,disponivel,disponivel,disponivel
1,M3,disponivel,disponivel,disponivel,disponivel,disponivel
2,M4,disponivel,disponivel,disponivel,disponivel,disponivel
3,M5,disponivel,disponivel,disponivel,disponivel,disponivel
4,intervalo,x,x,x,x,x
5,T1,disponivel,disponivel,disponivel,disponivel,disponivel
6,T2,disponivel,disponivel,disponivel,disponivel,disponivel
7,T3,disponivel,disponivel,disponivel,disponivel,disponivel
8,T4,disponivel,disponivel,disponivel,disponivel,disponivel
9,T5,disponivel,disponivel,disponivel,disponivel,disponivel


In [54]:

model = cp_model.CpModel()
dias = [2, 3, 4, 5, 6]
turnos_manha = ['m2', 'm3', 'm4', 'm5', 'm6']
turnos_tarde = ['t1', 't2', 't3', 't4', 't5'] # Turnos exatos dos seus blocos (sem t1 e t6)
horarios = turnos_manha + turnos_tarde
# Definindo as "Peças de Tetris" (Sem m6 ou t1 que não existem)
pares = [('m2','m3'), ('m4','m5'), ('t2','t3'), ('t4','t5')]
# Adicionando Trincas válidas para acomodar o Controle 1!
trincas = [('m4','m5','m6'), ('t1','t2','t3')]
quartetos = [('m2','m3','m4','m5'), ('t2','t3','t4','t5')]
# Dicionários de variáveis (Voltando com o y3)
x, y2, y3, y4, day_active = {}, {}, {}, {}, {}
is_morn_dict, is_aft_dict = {}, {}

In [55]:
# 1. CRIAR AS VARIÁVEIS BASE
for _, row in disciplinas.iterrows():
    disc = row['disciplina']
    is_morn_dict[disc] = model.NewBoolVar(f'morn_{disc}')
    is_aft_dict[disc] = model.NewBoolVar(f'aft_{disc}')
    model.AddExactlyOne([is_morn_dict[disc], is_aft_dict[disc]])
    
    for d in dias:
        day_active[(disc, d)] = model.NewBoolVar(f'da_{disc}_{d}')
        for h in horarios: x[(disc, d, h)] = model.NewBoolVar(f'x_{disc}_{d}_{h}')
        for i in range(len(pares)): y2[(disc, d, i)] = model.NewBoolVar(f'y2_{disc}_{d}_{i}')
        for i in range(len(trincas)): y3[(disc, d, i)] = model.NewBoolVar(f'y3_{disc}_{d}_{i}')
        for i in range(len(quartetos)): y4[(disc, d, i)] = model.NewBoolVar(f'y4_{disc}_{d}_{i}')

In [56]:
# 2. CONECTAR AS REGRAS ESTRUTURAIS 
for _, row in disciplinas.iterrows():
    disc = row['disciplina']
    for d in dias:
        model.Add(day_active[(disc, d)] == sum(y2[(disc, d, i)] for i in range(len(pares))) + 
                                         sum(y3[(disc, d, i)] for i in range(len(trincas))) + 
                                         sum(y4[(disc, d, i)] for i in range(len(quartetos))))
        for h in horarios:
            match_y2 = [y2[(disc, d, i)] for i, par in enumerate(pares) if h in par]
            match_y3 = [y3[(disc, d, i)] for i, tr in enumerate(trincas) if h in tr]
            match_y4 = [y4[(disc, d, i)] for i, qu in enumerate(quartetos) if h in qu]
            model.Add(x[(disc, d, h)] == sum(match_y2) + sum(match_y3) + sum(match_y4))
            
            if h in turnos_manha: model.Add(x[(disc, d, h)] == 0).OnlyEnforceIf(is_aft_dict[disc])
            if h in turnos_tarde: model.Add(x[(disc, d, h)] == 0).OnlyEnforceIf(is_morn_dict[disc])


In [57]:
# 3. REGRAS DE ESPAÇAMENTO DE DIAS (Evita Segunda e Sexta, etc)
# Padrões permitidos na semana: 1 = Tem aula no dia, 0 = Não tem. (Seg, Ter, Qua, Qui, Sex)
for _, row in disciplinas.iterrows():
    disc = row['disciplina']
    ch = row['carga horária']
    blocado = str(row['blocado']).strip()
    
    # Variáveis para travar o "Slot Fixo" (Mesmo horário todos os dias)
    pair_chosen = [model.NewBoolVar(f'pc_{disc}_{i}') for i in range(len(pares))]
    trinca_chosen = [model.NewBoolVar(f'tc_{disc}_{i}') for i in range(len(trincas))]
    
    if ch == 30:
        model.Add(sum(day_active[(disc, d)] for d in dias) == 1)
        model.AddExactlyOne(pair_chosen)
        for i in range(len(pares)):
            model.Add(sum(y2[(disc, d, i)] for d in dias) == 1).OnlyEnforceIf(pair_chosen[i])
            model.Add(sum(y2[(disc, d, i)] for d in dias) == 0).OnlyEnforceIf(pair_chosen[i].Not())
        model.Add(sum(y3[(disc, d, i)] for d in dias for i in range(len(trincas))) == 0)
        model.Add(sum(y4[(disc, d, i)] for d in dias for i in range(len(quartetos))) == 0)
        
    elif ch == 60:
        usa_quarteto = False
        if blocado != 'não' and blocado != 'nan':
            for parte in blocado.split(';'):
                if re.match(r'(\d)?([a-zA-Z])(\d{4})', parte.strip()): usa_quarteto = True
                    
        if usa_quarteto:
            model.Add(sum(day_active[(disc, d)] for d in dias) == 1)
            model.Add(sum(y4[(disc, d, i)] for d in dias for i in range(len(quartetos))) == 1)
            model.Add(sum(y2[(disc, d, i)] for d in dias for i in range(len(pares))) == 0)
        else:
            model.Add(sum(day_active[(disc, d)] for d in dias) == 2)
            # Regra Slot Fixo
            model.AddExactlyOne(pair_chosen)
            for i in range(len(pares)):
                model.Add(sum(y2[(disc, d, i)] for d in dias) == 2).OnlyEnforceIf(pair_chosen[i])
                model.Add(sum(y2[(disc, d, i)] for d in dias) == 0).OnlyEnforceIf(pair_chosen[i].Not())
            model.Add(sum(y4[(disc, d, i)] for d in dias for i in range(len(quartetos))) == 0)
            
            # Espaçamento de dias (não pode seg-qui, seg-sex, ter-sex)
            model.Add(day_active[(disc, 2)] + day_active[(disc, 5)] <= 1)
            model.Add(day_active[(disc, 2)] + day_active[(disc, 6)] <= 1)
            model.Add(day_active[(disc, 3)] + day_active[(disc, 6)] <= 1)
            
        model.Add(sum(y3[(disc, d, i)] for d in dias for i in range(len(trincas))) == 0)
        
    elif ch == 90:
        # TODAS as disciplinas de 90h usam Trincas travadas no mesmo horário!
        model.Add(sum(day_active[(disc, d)] for d in dias) == 2)
        model.AddExactlyOne(trinca_chosen)
        for i in range(len(trincas)):
            model.Add(sum(y3[(disc, d, i)] for d in dias) == 2).OnlyEnforceIf(trinca_chosen[i])
            model.Add(sum(y3[(disc, d, i)] for d in dias) == 0).OnlyEnforceIf(trinca_chosen[i].Not())
        model.Add(sum(y2[(disc, d, i)] for d in dias for i in range(len(pares))) == 0)
        model.Add(sum(y4[(disc, d, i)] for d in dias for i in range(len(quartetos))) == 0)
        
        # Espaçamento de dias 
        model.Add(day_active[(disc, 2)] + day_active[(disc, 5)] <= 1)
        model.Add(day_active[(disc, 2)] + day_active[(disc, 6)] <= 1)
        model.Add(day_active[(disc, 3)] + day_active[(disc, 6)] <= 1)


In [58]:
# 4. TRANSBORDAMENTO, TURNOS ALTERNADOS E CHOQUES
bonus_objetivo = []
for periodo in disciplinas['periodo'].unique():
    p_morn = model.NewBoolVar(f'pmorn_{periodo}')
    p_aft = model.NewBoolVar(f'paft_{periodo}')
    model.AddExactlyOne([p_morn, p_aft]) 
    
    # REGRA DE ALTERNÂNCIA: Semestres Pares à Tarde, Ímpares de Manhã
    if periodo % 2 == 0:
        #model.Add(p_aft == 1) 
        model.Add(p_morn == 1) 
    else:
        #model.Add(p_morn == 1)
        model.Add(p_aft == 1) 
        
    disc_do_periodo = disciplinas[disciplinas['periodo'] == periodo]['disciplina'].tolist()
        
    for disc in disc_do_periodo:
        if 'optativa' not in disc.lower():
            match_shift = model.NewBoolVar(f'match_{disc}')
            model.Add(is_morn_dict[disc] == p_morn).OnlyEnforceIf(match_shift)
            model.Add(is_morn_dict[disc] != p_morn).OnlyEnforceIf(match_shift.Not())
            bonus_objetivo.append(match_shift)
            
    # Choque de Período
    for d in dias:
        for h in horarios:
            model.AddAtMostOne(x[(d_p, d, h)] for d_p in disc_do_periodo)
# Choque E Agrupamento de Professor
prof_day_active = {}
for prof in disciplinas['professor'].unique():
    if prof != 'nda':
        disc_do_prof = disciplinas[disciplinas['professor'] == prof]['disciplina'].tolist()
        for d in dias:
            # Regra rígida: Professor não pode dar duas aulas ao mesmo tempo
            for h in horarios:
                model.AddAtMostOne(x[(d_p, d, h)] for d_p in disc_do_prof)
                
            # Cria a variável que "apita" (fica 1) se o professor for na faculdade neste dia
            prof_day_active[(prof, d)] = model.NewBoolVar(f'pda_{prof}_{d}')
            model.AddMaxEquality(prof_day_active[(prof, d)], [day_active[(d_p, d)] for d_p in disc_do_prof])
# FUNÇÃO OBJETIVO DUPLA MESTRE: 
# 1. Maximiza alunos no turno correto (Peso 10 - Prioridade Alta)
# 2. Minimiza as idas semanais do professor à faculdade (Peso 1)
model.Maximize(sum(bonus_objetivo) * 10 - sum(prof_day_active.values()))
# Horários Blocados do Excel
for _, row in disciplinas.iterrows():
    disc = row['disciplina']
    blocado = str(row['blocado']).strip()
    if blocado != 'não' and blocado != 'nan':
        for parte in blocado.split(';'):
            match = re.match(r'(\d)?([a-zA-Z])(\d+)', parte.strip())
            if match:
                dia_str, turno, slots = match.groups()
                if dia_str:
                    dia = int(dia_str)
                    for slot in slots:
                        h = f"{turno.lower()}{slot}"
                        if h in horarios and dia in dias:
                            model.Add(x[(disc, dia, h)] == 1)

bonus_objetivo = []
for periodo in disciplinas['periodo'].unique():
    p_morn = model.NewBoolVar(f'pmorn_{periodo}')
    p_aft = model.NewBoolVar(f'paft_{periodo}')
    model.AddExactlyOne([p_morn, p_aft]) 
    
    disc_do_periodo = disciplinas[disciplinas['periodo'] == periodo]['disciplina'].tolist()
    
    # Forçar turno preferencial do 8º Período para Tarde
    if periodo == 8:
        model.Add(p_aft == 1)
        
    for disc in disc_do_periodo:
        if 'optativa' not in disc.lower():
            match_shift = model.NewBoolVar(f'match_{disc}')
            model.Add(is_morn_dict[disc] == p_morn).OnlyEnforceIf(match_shift)
            model.Add(is_morn_dict[disc] != p_morn).OnlyEnforceIf(match_shift.Not())
            bonus_objetivo.append(match_shift)
            
    # Choque de Período
    for d in dias:
        for h in horarios:
            model.AddAtMostOne(x[(d_p, d, h)] for d_p in disc_do_periodo)
model.Maximize(sum(bonus_objetivo))
# Choque de Professor
for prof in disciplinas['professor'].unique():
    if prof != 'nda':
        disc_do_prof = disciplinas[disciplinas['professor'] == prof]['disciplina'].tolist()
        for d in dias:
            for h in horarios:
                model.AddAtMostOne(x[(d_p, d, h)] for d_p in disc_do_prof)
# Horários Blocados do Excel
for _, row in disciplinas.iterrows():
    disc = row['disciplina']
    blocado = str(row['blocado']).strip()
    if blocado != 'não' and blocado != 'nan':
        for parte in blocado.split(';'):
            match = re.match(r'(\d)?([a-zA-Z])(\d+)', parte.strip())
            if match:
                dia_str, turno, slots = match.groups()
                if dia_str:
                    dia = int(dia_str)
                    for slot in slots:
                        h = f"{turno.lower()}{slot}"
                        if h in horarios and dia in dias:
                            model.Add(x[(disc, dia, h)] == 1)

In [59]:
solver = cp_model.CpSolver()
solver.parameters.max_time_in_seconds = 120.0 
status = solver.Solve(model)
if status == cp_model.OPTIMAL or status == cp_model.FEASIBLE:
    print(f"Sucesso! Status: {solver.StatusName(status)}")
    print(f"Disciplinas agrupadas perfeitamente no turno: {int(solver.ObjectiveValue())} de {len(bonus_objetivo)}")
else:
    print("Infeasible: As restrições entraram em conflito.")

Sucesso! Status: OPTIMAL
Disciplinas agrupadas perfeitamente no turno: 127 de 20


In [60]:
# 1. Extrair os resultados onde o solver colocou "1" (aula acontece)
resultados = []
for _, row in disciplinas.iterrows():
    disc = row['disciplina']
    periodo = row['periodo']
    for d in dias:
        for h in horarios:
            # Se o solver decidiu que essa variável é Verdadeira (1)
            if solver.Value(x[(disc, d, h)]) == 1:
                resultados.append({
                    'Dia': d,
                    'Horário': h,
                    'Período': periodo,
                    'Disciplina': disc
                })

# 2. Converter para um DataFrame do Pandas
df_resultados = pd.DataFrame(resultados)

# 3. Mapear os números dos dias para nomes reais
mapa_dias = {2: 'SEGUNDA', 3: 'TERÇA', 4: 'QUARTA', 5: 'QUINTA', 6: 'SEXTA'}
df_resultados['Dia_Nome'] = df_resultados['Dia'].map(mapa_dias)

# 4. Gerar e exibir uma tabela bonitinha (Pivot Table) para cada Período
from IPython.display import display, HTML

for periodo in sorted(df_resultados['Período'].unique()):
    print(f"\n{'='*50}")
    print(f"GRADE DO {periodo}º PERÍODO")
    print(f"{'='*50}")
    
    # Filtra apenas as aulas deste semestre
    df_periodo = df_resultados[df_resultados['Período'] == periodo]
    
    # Cria a tabela cruzada: Linhas = Horário, Colunas = Dias, Valores = Disciplina
    grade = df_periodo.pivot(index='Horário', columns='Dia_Nome', values='Disciplina')
    
    # Garante que os dias fiquem na ordem correta (Segunda a Sexta)
    colunas_presentes = [mapa_dias[d] for d in dias if mapa_dias[d] in grade.columns]
    grade = grade[colunas_presentes]
    
    # Garante que as linhas fiquem na ordem correta (m2 até t6)
    linhas_presentes = [h for h in horarios if h in grade.index]
    grade = grade.reindex(linhas_presentes)
    
    # Troca os "NaN" (horários vazios) por um tracinho e exibe a tabela formatada
    grade = grade.fillna('---')
    
    # No Jupyter, usar o display mostra a tabela HTML bonitinha do Pandas
    display(grade)



GRADE DO 6º PERÍODO


Dia_Nome,QUARTA,QUINTA,SEXTA
Horário,,,
m2,lab de circuitos digitais,circuitos digitais,circuitos digitais
m3,lab de circuitos digitais,circuitos digitais,circuitos digitais
m4,---,sinais e sistemas,sinais e sistemas
m5,---,sinais e sistemas,sinais e sistemas
m6,---,sinais e sistemas,sinais e sistemas



GRADE DO 7º PERÍODO


Dia_Nome,SEGUNDA,TERÇA,QUARTA,QUINTA,SEXTA
Horário,,,,,
m2,sistemas distribuidos,digitais embarcados,digitais embarcados,---,---
m3,sistemas distribuidos,digitais embarcados,digitais embarcados,---,---
m4,sistemas distribuidos,---,---,---,---
m5,sistemas distribuidos,---,---,---,---
t1,controle 1,---,controle 1,---,---
t2,controle 1,---,controle 1,poo,poo
t3,controle 1,---,controle 1,poo,poo
t4,teoria da computação,---,teoria da computação,std,std
t5,teoria da computação,---,teoria da computação,std,std



GRADE DO 8º PERÍODO


Dia_Nome,SEGUNDA,TERÇA,QUARTA,QUINTA,SEXTA
Horário,,,,,
m2,---,lab de eletrônica analógica,controle 2,---,controle 2
m3,---,lab de eletrônica analógica,controle 2,---,controle 2
m4,---,eletrônica analógica,eletrônica analógica,sistemas inteligentes,sistemas inteligentes
m5,---,eletrônica analógica,eletrônica analógica,sistemas inteligentes,sistemas inteligentes
t2,engenharia de software,---,engenharia de software,---,dependabilidade e segurança
t3,engenharia de software,---,engenharia de software,---,dependabilidade e segurança
t4,banco de dados,---,banco de dados,---,dependabilidade e segurança
t5,banco de dados,---,banco de dados,---,dependabilidade e segurança



GRADE DO 9º PERÍODO


Dia_Nome,TERÇA,QUARTA,QUINTA,SEXTA
Horário,,,,
t2,processamento digital de sinais,processamento digital de sinais,instrumentação,instrumentação
t3,processamento digital de sinais,processamento digital de sinais,instrumentação,instrumentação
t4,segurança do trabalho,automação industrial,segurança do trabalho,automação industrial
t5,segurança do trabalho,automação industrial,segurança do trabalho,automação industrial
